# Unit 4 — LLM RL Foundations & Reward Models

> *Part of the [RL for Robotics & LLMs](https://github.com/AliBuildsAI/rl-for-robotics-llms) series.*

Units 1–3 built the full policy-gradient stack — REINFORCE → VPG → A2C+GAE →
PPO — on classic control environments where the **environment hands you a
reward** at every step (`+1` for balancing, `−1` per timestep, `+100` for
landing).

For language models there is no such function.  You cannot write code that
scores whether a paragraph is *helpful*, *honest*, or *harmless*.  This is the
single biggest change going from robotics RL to LLM RL, and it is what this
unit solves.

**RLHF** (Reinforcement Learning from Human Feedback) replaces the missing
reward function with a **reward model** $r_\phi$ — a neural network trained on
human preference comparisons.  Once we have $r_\phi$, Units 5–7 will optimise
a language model against it using exactly the policy gradients we already know.

**What we build in this unit:**

| Part | What | Outcome |
|------|------|---------|
| A | Load a base LM + preference data | See what "chosen vs rejected" looks like |
| B | Train a reward model with the Bradley-Terry loss (TRL) | A network that scores responses |
| C | Evaluate + probe | Preference accuracy, and watch it rank hand-written answers |

The reward model trained here is the **artifact Unit 5 consumes** when we run
PPO on a language model.


---
## 1  From Environments to Language — The MDP, Reframed

Everything in Units 1–3 was a Markov Decision Process: state $s_t$, action
$a_t$, reward $r_t$, transition $s_{t+1} \sim p(\cdot|s_t, a_t)$.  Language
generation is *also* an MDP — but a peculiar one.  Mapping the pieces over is
the key mental model for the rest of the series.

| MDP piece | Robotics (Units 1–3) | Language model |
|-----------|----------------------|----------------|
| **State** $s_t$ | Cart position, pole angle… | The prompt + all tokens generated so far |
| **Action** $a_t$ | 1 of 4 engines | 1 of ~150,000 vocabulary tokens |
| **Policy** $\pi_\theta(a_t\mid s_t)$ | Small MLP | The LM itself (softmax over vocab) |
| **Transition** $p(s_{t+1}\mid s_t,a_t)$ | box2d physics | **Deterministic** — just append the token |
| **Reward** $r_t$ | Given by the env | **Unknown** — must be learned |
| **Episode** | Until pole falls / lander lands | Until the `<eos>` token |

### Five structural differences worth internalising

1. **The action space is enormous and discrete.** A `Categorical` over 150k
   logits, not 4.  Same math as Unit 1's policy gradient, vastly wider head.

2. **Transitions are trivial.** $s_{t+1}$ is just $s_t$ with one token
   appended — deterministic and known.  This is why the dynamics cancel so
   cleanly in importance sampling (Unit 3, §4): there is literally nothing
   stochastic in the transition.

3. **Reward is terminal and sparse.** You only find out if a *whole* response
   was good after `<eos>`.  There is one scalar reward per generation, assigned
   to the final token — closer to Acrobot's "−1 until you succeed" than to
   LunarLander's dense shaping.  This is the credit-assignment problem on hard
   mode, and KL regularisation (Unit 5) is what keeps it stable.

4. **The policy is not initialised randomly.** It starts as a pretrained,
   instruction-tuned LM that already produces fluent text.  RL only *nudges*
   it.  We must keep it from drifting into gibberish — hence a reference model
   and a KL anchor, which had no analogue in robotics.

5. **The reward is learned, not given.** Which is this unit.

This last point is the whole game.  Let's build the reward.


---
## 2  Where Does Reward Come From? — Preferences, Not Scores

The naive idea: ask humans to rate responses 1–10 and regress against the
score.  This fails in practice — absolute ratings are noisy and
**uncalibrated** across annotators (my 7 is your 5).

The reliable signal is **comparison**.  Show a human a prompt $x$ and two
responses; they pick the better one.  Relative judgements are far more
consistent than absolute scores.  So our data is a set of triples:

$$(x,\; y_w,\; y_l) \quad\text{where } y_w \text{ is preferred (won) over } y_l \text{ (lost)}$$

We now need a model that turns these *discrete comparisons* into a
*continuous* scalar reward $r_\phi(x, y)$.  The bridge is a classical model of
pairwise choice.

### The Bradley-Terry model

Bradley-Terry (1952) assigns each item a latent strength and says the
probability one beats another is a logistic function of their strength
*difference*.  In our notation, with $r_\phi(x,y)$ as the latent strength:

$$P(y_w \succ y_l \mid x) =
\frac{\exp r_\phi(x, y_w)}{\exp r_\phi(x, y_w) + \exp r_\phi(x, y_l)}$$

Divide top and bottom by $\exp r_\phi(x, y_w)$ and this collapses to a sigmoid
of the difference:

$$P(y_w \succ y_l \mid x) = \sigma\bigl(r_\phi(x, y_w) - r_\phi(x, y_l)\bigr)$$

where $\sigma(z) = 1/(1 + e^{-z})$.

**Read this carefully — it is the heart of reward modelling.** Only the
*difference* of rewards is constrained by data.  The absolute scale is free
(add a constant to every reward and probabilities are unchanged).  This is why
reward-model outputs are only meaningful *relative to each other*, never as
absolute quality scores — a fact that matters when we feed them into PPO.

### From probability to loss

We fit $\phi$ by maximum likelihood: maximise the probability the model
assigns to the human's actual choice.  Maximising log-likelihood = minimising
negative log-likelihood, so the **reward-model loss** is:

$$L(\phi) = -\,\mathbb{E}_{(x, y_w, y_l)}\Bigl[
  \log \sigma\bigl(r_\phi(x, y_w) - r_\phi(x, y_l)\bigr)
\Bigr]$$

That is the entire objective.  Minimise it and the model learns to assign
higher scalar reward to responses humans prefer.  Notice it is just **binary
cross-entropy on the reward difference** — nothing exotic.


---
## 3  Reward Model Architecture — An LM Backbone + a Scalar Head

How do we get a scalar $r_\phi(x, y)$ out of a transformer?

1. Concatenate prompt and response into one token sequence $[x; y]$.
2. Run it through a pretrained transformer backbone → a hidden vector per
   token, shape `(seq_len, hidden_dim)`.
3. Attach a **single linear layer** `hidden_dim → 1` (the "value head").
4. Read off the scalar at the position of the **last token** — that position
   has attended to the entire sequence, so it summarises the whole response.

That is exactly what `AutoModelForSequenceClassification` with `num_labels=1`
gives you: the pretrained LM with the language-modelling head swapped for a
1-output classification head.

$$r_\phi(x, y) = w^\top h_{\text{last}}(x, y) + b
\qquad h_{\text{last}} \in \mathbb{R}^{d_{\text{model}}},\; w \in \mathbb{R}^{d_{\text{model}}}$$

The backbone is initialised from the pretrained LM (it already understands
language); only the head is new.  Training fine-tunes both.

> **Connection to Units 1–3.** The critic in A2C/PPO mapped a state to a scalar
> $V(s)$ with a linear head.  A reward model maps a (prompt, response) to a
> scalar $r(x,y)$ with a linear head.  Same shape of object — different target
> (human preference instead of discounted return).


---
## 4  Setup

Unlike Units 1–3, **the GPU genuinely matters here** — the bottleneck is now
transformer forward/backward passes, not CPU physics.

**A free Colab T4 is enough for this unit** — training a 0.5B reward model on
4k pairs takes ~15–25 min. The next cell auto-detects precision: bf16 on
A100/L4, fp16 on T4 (the T4 has no bf16 support, so this matters). An A100 is
faster but not required here; you'll want it for the PPO/GRPO units later.


In [ ]:
%pip install -q "transformers>=4.45" "trl>=0.12" "datasets>=3.0" accelerate
print("All packages ready.")


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}  |  device: {device}")
if device.type != "cuda":
    print("WARNING: no GPU detected — training will be slow. "
          "Runtime → Change runtime type → GPU.")


### Base model

We use **Qwen2.5-0.5B-Instruct** — small enough to train a reward model in
minutes on a single GPU, capable enough that the preferences it learns are
meaningful.  Everything below is identical for a larger backbone
(`Qwen/Qwen2.5-1.5B-Instruct`, `TinyLlama/TinyLlama-1.1B-Chat-v1.0`); only the
runtime changes.

We start from the **Instruct** checkpoint (already instruction-tuned) so the
backbone understands chat formatting — this stands in for the "SFT model" step
of the RLHF pipeline.  The *same* base model will be the policy we optimise in
Units 5–7, so the whole series shares one model for direct comparison.


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Mixed-precision policy that works on any GPU:
#   A100/L4 (Ampere+) support bf16 → train in bf16 (stable, fast)
#   T4 (Turing) has NO bf16 → fall back to fp16 AMP, keep fp32 weights
#   (fp32 weights + fp16 autocast avoids the "unscale fp16 gradients" error)
use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
use_fp16 = device.type == "cuda" and not use_bf16
load_dtype = torch.bfloat16 if use_bf16 else torch.float32
print(f"bf16: {use_bf16}  |  fp16: {use_fp16}  |  weights: {load_dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token   # Qwen has a pad token; this is a safety net

# num_labels=1  →  the classification head outputs a single scalar = the reward
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    torch_dtype=load_dtype,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME}")
print(f"Parameters: {n_params/1e6:.0f}M")
print(f"Reward head: {model.score}")   # the new hidden_dim → 1 linear layer


### What `AutoModelForSequenceClassification` actually did

When that call ran, it printed a warning like *"Some weights … `['score.weight']`
were newly initialized … you should probably TRAIN this model."*  That warning is
**expected** — here's exactly what happened under the hood.

**It kept the backbone, dropped the LM head, and bolted on a random scalar head.**

- The **pretrained backbone** (token embeddings + all transformer blocks + final
  norm) is loaded from the checkpoint — it already understands language.
- The original **language-modelling head** (`lm_head`, a `Linear: hidden → vocab`,
  ~151k outputs) is **discarded** — we don't want next-token logits.
- A **brand-new** `Linear: hidden → 1` head (`model.score`) is created and
  **randomly initialised** (~0.02 std). For Qwen the hidden size is 896, so this
  head is just `896 × 1 = 896` parameters with no bias — tiny next to the 0.5B
  backbone.

In one line: *"same transformer, vocab-projection head replaced by a regression
head that outputs a scalar reward."*  This is exactly how Llama 2's reward models
were built (Touvron et al. 2023).

**How a whole sequence becomes one number.** The `score` head runs on *every*
token position → `(batch, seq_len, 1)`.  The model then reads off the logit at the
**last non-padding token** of each sequence, because in a causal transformer that
position has attended to the entire response.  This is why we set
`model.config.pad_token_id` above — without it, the model can't find where each
sequence really ends in a padded batch (and errors when `batch_size > 1`).

**Nothing is frozen.** By default every parameter — backbone *and* head — is
trainable, so `trainer.train()` full-fine-tunes the whole model.  Alternatives:
*linear probe* (freeze backbone, train only the head — cheap but weak), or *LoRA*
(adapters on a frozen base — the go-to when a big backbone won't fit in VRAM).


In [ ]:
# Verify the surgery: random head, no lm_head, nothing frozen ──────────────────
print("class           :", type(model).__name__)        # Qwen2ForSequenceClassification
print("has lm_head      :", hasattr(model, "lm_head"))   # False — it was dropped
print("score head       :", model.score)                 # Linear(896 → 1, bias=False)

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)
head  = sum(p.numel() for p in model.score.parameters())
print(f"\ntotal params    : {total/1e6:.1f}M")
print(f"trainable params: {train/1e6:.1f}M   (== total → nothing frozen)")
print(f"reward head     : {head} params   (the only randomly-initialised part)")

# The head starts random, so an untrained reward is meaningless noise:
print(f"\nscore.weight mean={model.score.weight.mean().item():+.4f} "
      f"std={model.score.weight.std().item():.4f}  ← small random init")


---
## Part A — The Preference Data

We use **UltraFeedback (binarised)** — a standard preference dataset where each
row is a prompt with a `chosen` and a `rejected` response, both in chat-message
format.  This is the same data family used to train models like Zephyr.

Each example is a triple $(x, y_w, y_l)$:
- `chosen`   — the response humans (or a strong judge) preferred → $y_w$
- `rejected` — the dispreferred response → $y_l$


In [ ]:
# trl-lib/ultrafeedback_binarized: columns 'chosen' and 'rejected',
# each a list of chat messages [{role, content}, ...].
dataset = load_dataset("trl-lib/ultrafeedback_binarized")

# Subsample for a fast, video-friendly run. Bump these up for a stronger model.
N_TRAIN = 4000
N_EVAL  = 500
train_ds = dataset["train"].select(range(N_TRAIN))
eval_ds  = dataset["test"].select(range(N_EVAL))

print(f"Train pairs: {len(train_ds)}")
print(f"Eval  pairs: {len(eval_ds)}")
print(f"Columns: {train_ds.column_names}")


In [ ]:
# Look at one preference pair so we know exactly what the model learns from.
ex = train_ds[0]
prompt = ex["chosen"][0]["content"]          # the shared user prompt
chosen_reply   = ex["chosen"][-1]["content"]  # preferred assistant reply
rejected_reply = ex["rejected"][-1]["content"]

def show(title, text, n=400):
    print(f"\n{'='*70}\n{title}\n{'='*70}")
    print(text[:n] + ("…" if len(text) > n else ""))

show("PROMPT (x)", prompt)
show("CHOSEN  (y_w — preferred)",  chosen_reply)
show("REJECTED (y_l — dispreferred)", rejected_reply)


---
## Part B — Train the Reward Model

We already derived the loss in §2:

$$L(\phi) = -\,\mathbb{E}_{(x, y_w, y_l)}\Bigl[
  \log \sigma\bigl(r_\phi(x, y_w) - r_\phi(x, y_l)\bigr)\Bigr]$$

Before handing it to TRL, let's compute this loss **by hand once** so there is
no magic.  We score a `chosen` and a `rejected` reply with the (untrained)
model and apply the Bradley-Terry loss directly.


In [ ]:
def reward_score(prompt_text, reply_text):
    """Scalar reward r_φ(x, y): format → tokenize → backbone → last-token head."""
    messages = [
        {"role": "user", "content": prompt_text},
        {"role": "assistant", "content": reply_text},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=1024).to(device)
    with torch.no_grad():
        # logits shape (1, 1) — the single scalar from the reward head
        return model(**inputs).logits[0, 0].float().item()

# Manual Bradley-Terry loss on one pair, untrained model
r_w = reward_score(prompt, chosen_reply)
r_l = reward_score(prompt, rejected_reply)
bt_loss = -F.logsigmoid(torch.tensor(r_w - r_l)).item()   # -log σ(r_w - r_l)

print(f"r(chosen)   = {r_w:+.3f}")
print(f"r(rejected) = {r_l:+.3f}")
print(f"margin r_w - r_l = {r_w - r_l:+.3f}   (we want this POSITIVE)")
print(f"Bradley-Terry loss = {bt_loss:.3f}")
print("\nUntrained: the margin is essentially random. Training pushes it positive.")


### Now the real training run with TRL

`RewardTrainer` implements exactly the loss above.  Given a dataset with
`chosen` and `rejected` columns, for each pair it:

1. tokenises both completions (applying the chat template),
2. forward-passes each through the model to get $r_\phi(x, y_w)$ and
   $r_\phi(x, y_l)$,
3. computes $-\log\sigma(r_w - r_l)$ and backpropagates.

We are using a stable implementation — but the loss it runs is the one we just
computed by hand.  (Same philosophy as Units 5+: we implemented PPO ourselves
in Unit 3, so from here we use vetted implementations and focus on understanding.)


In [ ]:
from trl import RewardTrainer, RewardConfig

config = RewardConfig(
    output_dir="qwen-rm",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=1e-5,
    logging_steps=25,
    eval_strategy="no",
    report_to="none",
    max_length=1024,
    bf16=use_bf16,    # A100/L4
    fp16=use_fp16,    # T4 fallback
    remove_unused_columns=False,
)

trainer = RewardTrainer(
    model=model,
    args=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

trainer.train()
print("\nReward model training complete.")


---
## Part C — Evaluate: Does It Rank Preferences Correctly?

The natural metric for a reward model is **preference accuracy**: on held-out
pairs, how often does it score the chosen response above the rejected one?

$$\text{accuracy} = \mathbb{E}_{(x,y_w,y_l)}
\bigl[\mathbb{1}\{\,r_\phi(x,y_w) > r_\phi(x,y_l)\,\}\bigr]$$

Random guessing is 50%.  A small model on UltraFeedback typically reaches
~65–72% — clearly above chance, which is all Unit 5 needs to get a useful
training signal.


In [ ]:
@torch.no_grad()
def preference_accuracy(ds, n=None):
    model.eval()
    n = n or len(ds)
    correct, margins = 0, []
    for i in range(n):
        ex = ds[i]
        p  = ex["chosen"][0]["content"]
        r_w = reward_score(p, ex["chosen"][-1]["content"])
        r_l = reward_score(p, ex["rejected"][-1]["content"])
        correct += (r_w > r_l)
        margins.append(r_w - r_l)
    model.train()
    return correct / n, float(np.mean(margins))

acc, mean_margin = preference_accuracy(eval_ds, n=200)
print(f"Held-out preference accuracy: {acc:6.1%}   (random = 50%)")
print(f"Mean reward margin r_w - r_l: {mean_margin:+.3f}   (want positive)")


### Is ~65% good? Yes — that's the production regime

Don't be underwhelmed by a number in the 60s; that is exactly where real reward
models live.  Meta's **shipped Llama 2 reward models** scored **63.2%**
(helpfulness) and **64.5%** (safety) on their own in-distribution test sets — and
that *beat GPT-4* as a judge (~58%).

The aggregate looks low because it averages over easy and hard pairs.  Llama 2
(Table 8) broke accuracy down by how separable the two responses are:

| Pair difficulty | Llama 2 RM accuracy |
|---|---|
| "Significantly better" | ~90–94% |
| "Negligibly better / unsure" | ~54–55% (near chance) |

So the model is *good* at obvious cases; the ~65% average is dragged down by
genuine coin-flips.  Curated benchmarks like **RewardBench** report ~90%+ for the
best models (Nemotron-4-340B-Reward 92.2, ArmoRM 90.8) — but that is a different,
easier metric built from cleanly-separable pairs, not in-distribution accuracy.

**Why we share the backbone with the policy.** Llama 2 initialised its reward
model *from the chat checkpoint* so the RM has the same knowledge as the policy —
otherwise the policy can reward-hack the RM's blind spots (e.g. hallucinate facts
the RM can't verify).  We do the same: our RM and the Unit 5 policy share the Qwen
backbone.

The next cell reproduces the Llama 2 Table 8 pattern on *our* model, using
UltraFeedback's own quality scores to label each pair "clear" or "close".


In [ ]:
# Accuracy by preference clarity — reproduces Llama 2's Table 8 on our model ─────
# UltraFeedback ships a numeric quality score per response; the gap between them
# tells us how "separable" the pair is. We expect: high accuracy on clear pairs,
# near-chance on close ones.
@torch.no_grad()
def accuracy_by_clarity(ds, n=200):
    if "score_chosen" not in ds.column_names:
        print("(dataset has no per-response scores — skipping this analysis)")
        return
    model.eval()
    gaps, hits = [], []
    for i in range(n):
        ex = ds[i]
        p   = ex["chosen"][0]["content"]
        r_w = reward_score(p, ex["chosen"][-1]["content"])
        r_l = reward_score(p, ex["rejected"][-1]["content"])
        gaps.append(ex["score_chosen"] - ex["score_rejected"])  # dataset's own margin
        hits.append(bool(r_w > r_l))
    model.train()

    gaps, hits = np.array(gaps), np.array(hits)
    med = np.median(gaps)
    clear, close = hits[gaps >= med], hits[gaps < med]
    print(f"Overall accuracy            : {hits.mean():6.1%}  (n={len(hits)})")
    print(f"Clear pairs  (score gap ≥ {med:.1f}): {clear.mean():6.1%}  (n={len(clear)})")
    print(f"Close pairs  (score gap < {med:.1f}): {close.mean():6.1%}  (n={len(close)})")
    print("\nSame shape as Llama 2 Table 8: the model is confident on clearly-"
          "\nseparable pairs and near chance on near-ties. The headline number"
          "\naverages over both.")

accuracy_by_clarity(eval_ds, n=200)


### The probe — watch it score hand-written answers

Accuracy is a number; this is the part that makes the reward model *feel* real.
We write our own prompt with an obviously good and an obviously bad answer and
print the two scalar rewards.  A trained reward model should give the helpful
answer a clearly higher score.


In [ ]:
probe_prompt = "Explain what a hash table is to a beginner."

good = ("A hash table stores key–value pairs. It runs the key through a hash "
        "function to compute an index into an array, so lookups, inserts, and "
        "deletes are on average O(1). Collisions — two keys landing on the same "
        "index — are handled by chaining or open addressing.")

bad  = "idk just use a list lol. hash tables are too complicated and useless."

r_good = reward_score(probe_prompt, good)
r_bad  = reward_score(probe_prompt, bad)

print(f"Prompt: {probe_prompt}\n")
print(f"  r(good answer) = {r_good:+.3f}")
print(f"  r(bad  answer) = {r_bad:+.3f}")
print(f"  margin         = {r_good - r_bad:+.3f}")
print("\n→ " + ("Reward model prefers the GOOD answer. ✓"
                 if r_good > r_bad else
                 "Reward model got this one wrong — try more training data."))


---
## 5  What's Next — PPO on a Language Model (Unit 5)

We now have the missing piece: a function $r_\phi(x, y)$ that scores any
response, learned entirely from human preference comparisons.  This is the
reward the environment refused to give us.

In **Unit 5** we plug this reward model into PPO — the exact algorithm from
Unit 3 — to fine-tune the language model itself.  The objective becomes:

$$\max_\theta\; \mathbb{E}_{x,\,y \sim \pi_\theta}\Bigl[
  r_\phi(x, y) - \beta\,\text{KL}\bigl(\pi_\theta(\cdot|x)\,\|\,\pi_{\text{ref}}(\cdot|x)\bigr)
\Bigr]$$

- $r_\phi$ — the reward model we just trained
- $\pi_{\text{ref}}$ — a frozen copy of the starting model, the **KL anchor**
  that keeps the policy from drifting into gibberish to hack the reward
- $\beta$ — how hard we pull back toward the reference

That is the full RLHF setup: **four models** (policy, reference, reward, value)
and the PPO update you already know.  We'll also see the failure mode that
motivates the KL term — set $\beta \to 0$ and the policy reward-hacks the
reward model into nonsense.

→ [Back to the series](https://github.com/AliBuildsAI/rl-for-robotics-llms)
